# Assessing Data Accessibility from Files Downloaded from the SEEA Knowledge Base
The data accessibility assessment uses files downloaded from the SEEA Knowledge Base: https://seea.un.org/content/knowledge-base
. The data scraping code can be found here: XXXX
Files are save in folder: **outputs_SEEAKB**

In [ ]:
import importlib.util
import json
import os
import re
import subprocess
import sys
import unicodedata
from pathlib import Path

REQUIRED_PACKAGES = {
    "pandas": "pandas",
    "openpyxl": "openpyxl",
    "bs4": "beautifulsoup4",
    "pypdf": "pypdf",
    "docx": "python-docx",
    "xlrd": "xlrd",
}

for import_name, package_name in REQUIRED_PACKAGES.items():
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing {package_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])

import pandas as pd

import run_seea_assessment_configured as seea


SOURCE_XLSX = "global_assessment_for_website_2024_final.xlsx"
SHEET_NAME = "2024 Global Assessment results"
CONFIG_JSON = "seea_keyword_config.json"
SEEAKB_FOLDER = "outputs_SEEAKB"

OUTPUT_XLSX = "availability_SEEAKB.xlsx"
FALLBACK_OUTPUT_XLSX = "availability_SEEAKB_updated.xlsx"
OUTPUT_WIDE_CSV = "availability_SEEAKB.csv"
OUTPUT_LONG_CSV = "availability_SEEAKB_long.csv"

COUNTRY_ALIASES = {
    "United Kingdom": "United Kingdom of Great Britain and Northern Ireland",
    "UK": "United Kingdom of Great Britain and Northern Ireland",
    "United States": "United States of America",
    "USA": "United States of America",
    "Korea": "Republic of Korea",
    "South Korea": "Republic of Korea",
    "Moldova": "Republic of Moldova",
    "Tanzania": "United Republic of Tanzania",
    "Turkey": "Türkiye",
    "Viet Nam": "Viet Nam",
    "Vietnam": "Viet Nam",
}


def normalize_label(value):
    value = str(value or "")
    value = unicodedata.normalize("NFKD", value)
    value = "".join(ch for ch in value if not unicodedata.combining(ch))
    value = value.lower()
    value = re.sub(r"[^a-z0-9]+", " ", value)
    return re.sub(r"\s+", " ", value).strip()


def infer_country_from_seeakb_folder(folder_name):
    parts = [part.strip() for part in str(folder_name).split(" - ")]
    if len(parts) >= 3 and re.fullmatch(r"\d{4}", parts[-1]):
        country = parts[-2]
    elif len(parts) >= 2:
        country = parts[-1]
    else:
        country = "Unknown"
    return COUNTRY_ALIASES.get(country, country)


def load_seeakb_file_index(base_folder):
    base = Path(base_folder)
    records = []
    if not base.exists():
        raise FileNotFoundError(f"Missing folder: {base_folder}")

    for item_folder in sorted(path for path in base.iterdir() if path.is_dir()):
        country = infer_country_from_seeakb_folder(item_folder.name)
        for file_path in item_folder.rglob("*"):
            if file_path.is_file():
                records.append(
                    {
                        "country": country,
                        "seeakb_item": item_folder.name,
                        "file_path": file_path,
                    }
                )
    return records


def build_keyword_cache(countries, config):
    keyword_cache = {}
    keyword_rows = []
    for country in countries:
        country_languages = config["country_languages"].get(country, ["English"])
        for account_type in seea.ACCOUNT_TYPES:
            phrases = seea.generated_keyword_phrases(country, account_type, config)
            rows = []
            for phrase in phrases:
                row = {
                    "country": country,
                    "account_type": account_type,
                    "language": "|".join(country_languages),
                    "keyword_phrase": phrase,
                    "normalized_keyword": seea.normalize_text(phrase),
                }
                rows.append(row)
                keyword_rows.append(row)
            keyword_cache[(country, account_type)] = rows
    return keyword_cache, keyword_rows


TOPIC_STOPWORDS = {
    "account",
    "accounts",
    "accounting",
    "physical",
    "monetary",
    "asset",
    "assets",
    "stock",
    "stocks",
    "resource",
    "resources",
    "for",
    "and",
    "or",
    "of",
    "the",
    "in",
    "to",
    "by",
    "table",
    "tables",
    "supply",
    "use",
    "using",
    "with",
    "sector",
    "terms",
    "unit",
    "units",
    "data",
    "hybrid",
}


def match_account_in_file(
    text,
    normalized_text,
    filename,
    normalized_filename,
    keyword_rows,
    account_type,
    country,
    config,
    location_type,
):
    sources = [
        ("filename", filename, normalized_filename),
        ("content", text, normalized_text),
    ]
    for location, source_text, normalized_source in sources:
        for row in keyword_rows:
            normalized_keyword = row["normalized_keyword"]
            if normalized_keyword and normalized_keyword in normalized_source:
                snippet = seea.make_snippet(source_text, row["keyword_phrase"])
                if seea.has_false_positive_context(snippet):
                    continue
                return {
                    "matched_keyword": row["keyword_phrase"],
                    "match_location": location,
                    "snippet": snippet,
                }
        if location == "content" and location_type == "pdf text":
            toc_match = find_pdf_toc_account_match(source_text, account_type, country, config)
            if toc_match:
                return toc_match
    return None


def find_pdf_toc_account_match(source_text, account_type, country, config):
    account_words = get_account_words_for_country(country, config)
    topic_token_sets = get_account_topic_token_sets(account_type, country, config)
    if not account_words or not topic_token_sets:
        return None

    for line in extract_pdf_toc_candidate_lines(source_text):
        normalized_line = seea.normalize_text(line)
        if not normalized_line:
            continue
        if not has_account_word(normalized_line, account_words):
            continue
        if seea.has_false_positive_context(line):
            continue

        line_tokens = set(normalized_line.split())
        if not any(topic_tokens_match(line_tokens, token_set) for token_set in topic_token_sets):
            continue

        return {
            "matched_keyword": f"PDF TOC/front matter: {line.strip()}",
            "match_location": "pdf table of contents/front matter",
            "snippet": line.strip(),
        }
    return None


def get_account_words_for_country(country, config):
    languages = ["English", *config["country_languages"].get(country, [])]
    words = set()
    for language in languages:
        for word in config["account_words_by_language"].get(language, []):
            normalized_word = seea.normalize_text(word)
            if normalized_word:
                words.add(normalized_word)
    return words


def has_account_word(normalized_line, account_words):
    line_tokens = set(normalized_line.split())
    if line_tokens & account_words:
        return True
    account_phrases = {
        "account",
        "accounts",
        "accounting",
        "stock accounts",
        "flow accounts",
        "asset accounts",
        "resource accounts",
    }
    return any(phrase in normalized_line for phrase in account_phrases)


def get_account_topic_token_sets(account_type, country, config):
    languages = ["English", *config["country_languages"].get(country, [])]
    phrases = [account_type]
    topic_map = config["topic_translations"].get(account_type, {})
    for language in languages:
        phrases.extend(topic_map.get(language, []))

    official_tokens = phrase_to_topic_tokens(account_type)
    token_sets = []
    for phrase in phrases:
        phrase_tokens = phrase_to_topic_tokens(phrase)
        if not phrase_tokens:
            continue

        # When the official account name has multiple distinctive English terms
        # such as "mineral" + "energy", do not let a one-word English subset
        # such as only "energy" trigger the account. One-token non-English
        # phrases are kept because some languages are not space-delimited.
        if (
            len(official_tokens) > 1
            and len(phrase_tokens) == 1
            and phrase_tokens.issubset(official_tokens)
        ):
            continue

        if phrase_tokens and phrase_tokens not in token_sets:
            token_sets.append(phrase_tokens)
    return token_sets


def phrase_to_topic_tokens(phrase):
    normalized_phrase = seea.normalize_text(phrase)
    tokens = set()
    for token in normalized_phrase.split():
        if len(token) < 3:
            continue
        if token in TOPIC_STOPWORDS:
            continue
        tokens.add(token)
    return tokens


def topic_tokens_match(line_tokens, topic_tokens):
    if not topic_tokens:
        return False

    overlap = line_tokens & topic_tokens

    if len(topic_tokens) == 1:
        return len(overlap) == 1

    if len(topic_tokens) == 2:
        return len(overlap) == 2

    required = max(2, int(round(len(topic_tokens) * 0.7)))
    return len(overlap) >= required


def extract_pdf_toc_candidate_lines(source_text):
    raw_lines = [line.strip() for line in str(source_text or "").splitlines()]
    lines = [line for line in raw_lines if line]
    normalized_lines = [seea.normalize_text(line) for line in lines]

    toc_indices = [
        index
        for index, line in enumerate(normalized_lines[:250])
        if "table of contents" in line or line == "contents" or "list of tables" in line
    ]

    selected = []
    if toc_indices:
        for index in toc_indices:
            selected.extend(lines[index : index + 140])
    else:
        selected.extend(lines[:220])

    cleaned = []
    for line in selected:
        # Remove common dot leaders and terminal page numbers while preserving section text.
        line = re.sub(r"\.{3,}\s*\d+\s*$", "", line).strip()
        line = re.sub(r"\s+\d+\s*$", "", line).strip()
        if len(line) < 6:
            continue
        cleaned.append(line)

    return cleaned


def autosize_excel(writer):
    for worksheet in writer.book.worksheets:
        worksheet.freeze_panes = "A2"
        for col_cells in worksheet.columns:
            col_letter = col_cells[0].column_letter
            max_len = 0
            for cell in col_cells[:2000]:
                value = "" if cell.value is None else str(cell.value)
                max_len = max(max_len, len(value))
            worksheet.column_dimensions[col_letter].width = min(max(max_len + 2, 10), 60)


def main():
    with open(CONFIG_JSON, "r", encoding="utf-8") as file:
        config = json.load(file)

    source_df = pd.read_excel(SOURCE_XLSX, sheet_name=SHEET_NAME, header=1)
    source_df["Country or area"] = source_df["Country or area"].astype(str).str.strip()

    seeakb_files = load_seeakb_file_index(SEEAKB_FOLDER)
    files_by_country = {}
    for record in seeakb_files:
        files_by_country.setdefault(record["country"], []).append(record)

    excel_countries = list(source_df["Country or area"])
    all_countries = list(dict.fromkeys(excel_countries + sorted(files_by_country)))
    keyword_cache, keyword_rows = build_keyword_cache(all_countries, config)

    reported_lookup = source_df.set_index("Country or area", drop=False)
    matrix_rows = []
    long_rows = []
    evidence_rows = []
    error_rows = []
    files_scanned = 0

    for country in all_countries:
        excel_row = reported_lookup.loc[country] if country in reported_lookup.index else None
        country_records = files_by_country.get(country, [])
        available = {account_type: 0 for account_type in seea.ACCOUNT_TYPES}
        matrix_row = {"Country or area": country}

        for account_type in seea.ACCOUNT_TYPES:
            reported = 0
            if excel_row is not None:
                reported = 1 if str(excel_row.get(account_type, "")).strip().lower() == "x" else 0
            matrix_row[f"{account_type}_reported"] = reported
            matrix_row[f"{account_type}_available"] = 0
            matrix_row[f"{account_type}_nonreport_avail"] = 0

        for record in country_records:
            file_path = record["file_path"]
            files_scanned += 1
            try:
                text, location_type = seea.extract_file_text(file_path)
            except Exception as exc:
                error_rows.append(
                    {
                        "country": country,
                        "seeakb_item": record["seeakb_item"],
                        "file_path": str(file_path),
                        "error_message": str(exc),
                    }
                )
                continue

            filename = file_path.name
            normalized_filename = seea.normalize_text(filename)
            normalized_text = seea.normalize_text(text)

            for account_type in seea.ACCOUNT_TYPES:
                if available[account_type] == 1:
                    continue
                match = match_account_in_file(
                    text,
                    normalized_text,
                    filename,
                    normalized_filename,
                    keyword_cache[(country, account_type)],
                    account_type,
                    country,
                    config,
                    location_type,
                )
                if not match:
                    continue
                available[account_type] = 1
                evidence_rows.append(
                    {
                        "country": country,
                        "account_type": account_type,
                        "matched_keyword": match["matched_keyword"],
                        "matched_file": filename,
                        "file_path": str(file_path),
                        "seeakb_item": record["seeakb_item"],
                        "match_location": "filename" if match["match_location"] == "filename" else match["match_location"] if match["match_location"] != "content" else location_type,
                        "snippet": match["snippet"],
                    }
                )

        for account_type in seea.ACCOUNT_TYPES:
            reported = matrix_row[f"{account_type}_reported"]
            avail = available[account_type]
            nonreport = 1 if avail == 1 and reported == 0 else 0
            matrix_row[f"{account_type}_available"] = avail
            matrix_row[f"{account_type}_nonreport_avail"] = nonreport
            long_rows.append(
                {
                    "country": country,
                    "account_type": account_type,
                    "reported": reported,
                    "available": avail,
                    "nonreport_avail": nonreport,
                }
            )

        matrix_rows.append(matrix_row)
        print(f"{country}: SEEAKB files={len(country_records)} available_accounts={sum(available.values())}")

    matrix_df = pd.DataFrame(matrix_rows)
    long_df = pd.DataFrame(long_rows)
    evidence_df = pd.DataFrame(evidence_rows)
    errors_df = pd.DataFrame(error_rows)
    keyword_df = pd.DataFrame(keyword_rows).drop(columns=["normalized_keyword"], errors="ignore")

    matrix_df.to_csv(OUTPUT_WIDE_CSV, index=False, encoding="utf-8-sig")
    long_df.to_csv(OUTPUT_LONG_CSV, index=False, encoding="utf-8-sig")

    output_xlsx = OUTPUT_XLSX
    try:
        writer = pd.ExcelWriter(output_xlsx, engine="openpyxl")
    except PermissionError:
        output_xlsx = FALLBACK_OUTPUT_XLSX
        writer = pd.ExcelWriter(output_xlsx, engine="openpyxl")

    with writer:
        matrix_df.to_excel(writer, sheet_name="availability_matrix", index=False)
        evidence_df.to_excel(writer, sheet_name="availability_evidence", index=False)
        errors_df.to_excel(writer, sheet_name="file_read_errors", index=False)
        keyword_df.to_excel(writer, sheet_name="keyword_dictionary", index=False)
        autosize_excel(writer)

    print("\nDone.")
    print(f"Countries processed: {len(matrix_rows)}")
    print(f"SEEAKB files scanned: {files_scanned}")
    print(f"Evidence rows: {len(evidence_rows)}")
    print(f"File read errors: {len(error_rows)}")
    print(f"Excel output: {os.path.abspath(output_xlsx)}")
    print(f"Wide CSV output: {os.path.abspath(OUTPUT_WIDE_CSV)}")
    print(f"Long CSV output: {os.path.abspath(OUTPUT_LONG_CSV)}")


if __name__ == "__main__":
    main()
